# Task 2 — Match-Day Attendance: Knockout vs. Group Stage

## Analytic question formulation

**Is average stadium attendance at knockout-stage matches significantly
different from attendance at group-stage matches during the 2026 FIFA World
Cup?**

With 48 teams and an expanded format, group-stage matches were spread
across many double-headers and weekday slots, while knockout matches are
single, high-stakes fixtures often in larger marquee stadiums. This task
quantifies whether that intuition holds up statistically, using the
attendance of all 104 official matches of the tournament (the full
population of matches actually played).

**Skills demonstrated:** data wrangling → sampling → descriptive statistics
→ confidence interval → two-sample *t*-test.

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from data_prep import load_matches
from stats_utils import describe, ci_mean, two_sample_ttest, levene_test

np.random.seed(42)
pd.set_option("display.max_columns", None)

matches = load_matches()
matches[["match_number", "date", "stage", "team1", "team2", "attendance"]].head()

## Data wrangling

`matches_raw.csv` (compiled from FIFA.com match centre pages and
contemporaneous reports) holds one row per official match with its stage,
venue, teams, score, and reported attendance. `data_prep.load_matches()`:

1. Parses dates and adds an ordered `stage` category.
2. Adds a binary `knockout` flag (0 = Group Stage, 1 = Round of 32 through
   the Final) — this is the grouping variable for this task.
3. Adds `goal_diff`, used in Objective 2 but harmless here.

Every one of the 104 official matches has a reported attendance figure, so
unlike Task 1 there is no filtering step — the wrangling here is purely
about typing/labelling the stage variable correctly.

In [ ]:
print("Total matches:", matches.shape[0])
assert matches.shape[0] == 104, "Expected exactly 104 official matches"
print(matches["knockout"].value_counts().rename({0: "Group Stage", 1: "Knockout"}))
matches.groupby("stage", observed=True)["attendance"].agg(["count", "mean", "std"])

## Data preparation and sampling

**Population:** all matches of the 2026 FIFA World Cup, split into two
strata by `knockout` (Group Stage vs. Knockout Stage).

**Variable of interest:** `attendance` (reported crowd figure), one
observation per match.

**Sampling technique:** the knockout stage is a naturally small stratum
(32 matches vs. 72 group-stage matches), so a single unconstrained random
draw would risk an unbalanced, low-powered comparison. We instead use
**stratified sampling with equal allocation**: take the *entire* knockout
stratum (32 matches — small enough to use as a census) and a **simple
random sample of 32 matches, without replacement**, from the 72 group-stage
matches (fixed seed for reproducibility). This yields two balanced samples
of *n* = 32 each, adequate for a *t*-test and for a normal-approximation
confidence interval.

In [ ]:
group_pop = matches[matches["knockout"] == 0]
knockout_pop = matches[matches["knockout"] == 1]

knockout_sample = knockout_pop.copy()  # small stratum -> take full census
group_sample = group_pop.sample(n=min(32, len(group_pop)), random_state=42, replace=False)

print(f"Group-stage population: {len(group_pop)}  -> sample n={len(group_sample)}")
print(f"Knockout population:    {len(knockout_pop)}  -> sample n={len(knockout_sample)}")

## Descriptive statistics

In [ ]:
desc_table = pd.DataFrame(
    {
        "Group Stage": describe(group_sample["attendance"]),
        "Knockout Stage": describe(knockout_sample["attendance"]),
    }
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(
    [group_sample["attendance"], knockout_sample["attendance"]],
    tick_labels=["Group Stage", "Knockout"],
)
ax.set_ylabel("Attendance")
ax.set_title("Attendance by stage (sample)")
plt.tight_layout()
plt.savefig("../report/figs/task2_box.png", dpi=120)
plt.show()

desc_table

## Inferential statistics — confidence interval

We estimate a 95% confidence interval for the **population mean attendance
across all World Cup matches**, using our combined 64-match sample
(32 group + 32 knockout) as the working sample for the overall population
of matches, via the *t*-distribution.

In [ ]:
combined_sample = pd.concat([group_sample, knockout_sample])
ci = ci_mean(combined_sample["attendance"], confidence=0.95)
print(f"n = {ci['n']}")
print(f"Sample mean attendance = {ci['mean']:.0f}")
print(f"95% CI: ({ci['ci_low']:.0f}, {ci['ci_high']:.0f})")
ci

## Inferential statistics — two-sample *t*-test

$H_0: \mu_{knockout} = \mu_{group}$ (mean attendance equal between knockout
and group-stage matches)

$H_1: \mu_{knockout} \neq \mu_{group}$

We first check the equal-variance assumption with Levene's test, then run
Welch's two-sample *t*-test, at $\alpha = 0.05$.

In [ ]:
lev = levene_test(knockout_sample["attendance"], group_sample["attendance"])
print("Levene's test p-value:", lev["p_value"])

result = two_sample_ttest(
    knockout_sample["attendance"], group_sample["attendance"], equal_var=lev["p_value"] > 0.05
)
for k, v in result.items():
    print(f"{k}: {v}")

alpha = 0.05
if result["p_value"] < alpha:
    print(f"\nReject H0 (p={result['p_value']:.4f} < {alpha}): mean attendance differs significantly by stage.")
else:
    print(f"\nFail to reject H0 (p={result['p_value']:.4f} >= {alpha}): no significant difference detected.")

## Conclusion

*(Auto-filled after running the cells above with real data — summarize the
direction/magnitude of the attendance gap between knockout and group-stage
matches, its statistical significance at α=0.05, and the interpretation of
the overall attendance CI.)*